## EU REED

#### 1) Import packages, background databases and set project

In [13]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config

import matplotlib.pyplot as plt
import d3graph
import polyviz

In [14]:
config = Config()

In [15]:
bd.projects.set_current(config.PROJECT_NAME)

In [16]:
import import_db

#### 2) Import foreground database 

In [17]:
bonsai = bd.Database('bonsai_231231')
bonsai_bio = bd.Database('bonsai_231231 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [18]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

In [19]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260122.xlsx',)
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

Extracted 3 worksheets in 1.64 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [20]:
importer.statistics()

Graph statistics for `reed` importer:
25 graph nodes:
	None: 25
120 graph edges:
	technosphere: 92
	production: 25
	biosphere: 3
116 edges to the following databases:
	bonsai_231231: 66
	reed: 48
	ecoinvent-3.10-biosphere: 2
4 unique unlinked edges (4 total):
	technosphere: 3
	biosphere: 1




(25, 120, 4, 0)

In [21]:
importer.write_database()

 72%|███████▏  | 18/25 [00:00<00:00, 1999.99it/s]


InvalidExchange: 

In [22]:
pd.DataFrame(importer.unlinked)

,name,amount,unit,location,type,activity type,database,categories,reference product
0,compressed air,115.0000,kg,GB,technosphere,market activity,bonsai_231231,NaN,NaN
1,carbon dioxide,0.0266,kg,NaN,biosphere,NaN,ecoinvent-3.10-biosphere,air,NaN
2,milling,800.0000,kilogram,GLO,technosphere,NaN,bonsai_231231,NaN,"steel removed by millling, average"
3,"Treatment of steel waste, including sorting, c...",800.0000,kilogram,ES,technosphere,NaN,bonsai_231231,NaN,cast iron


In [ ]:
reed = bd.Database('reed')

In [ ]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [ ]:
from IPython.display import display
display(df_reed[['code', 'comment', 'location', 'unit', 'database']])

In [ ]:
bd.databases

#### 3) Select processes to be analysed

In [ ]:
lpg = bonsai.get(name = 'Liquefied Petroleum Gases (LPG), combustion of', location = "DE", unit = 'TJ')
steel = bonsai.get(name = 'basic iron and steel and of ferro-alloys and first products thereof', location = "DK", unit = 'tonne')
flamecutting = reed.get(name='flamecutting, lpg, meter', location="DE", unit='meter')
man_welding = reed.get(name='welding, manual', location="GB", unit='hour')
use_case_komatsu = reed.get(name='Komatsu piece', location="GB", unit='unit')

In [ ]:
check_activity_type = print(lpg["activity type"])

In [ ]:
machining_processes = {
    "man_welding": reed.get(name='welding, manual', location="GB", unit='hour'),
    "rob_welding": reed.get(name='welding, robotic', location="GB", unit='hour'),
    "drilling": reed.get(name='drilling', location="GB", unit='kilogram'),
    "bending": reed.get(name='bending', location="DE", unit='hour'),
    "sawing": reed.get(name='sawing', location="DE", unit='hour'),
    "turning": reed.get(name='turning', location="GB", unit='kilogram'),
    "flamecutting, lpg, meter": reed.get(name='flamecutting, lpg, meter', location="DE", unit='meter'),
    "flamecutting, natural gas,": reed.get(name='flamecutting, natural gas, meter', location="DE", unit='meter'),
}

In [ ]:
use_case_machines = {
    "soraluce_komatsu": reed.get(name='soraluce machine, Komatsu', location="GB", unit='unit'),
    "use_case_komatsu": reed.get(name='Komatsu piece', location="GB", unit='unit'),
    "soraluce_wwr": reed.get(name='soraluce machine, WWR', location="DE", unit='unit'),
    "use_case_wwr": reed.get(name='WWR piece', location="DE", unit='unit'),
    "soraluce_goimek": reed.get(name='soraluce machine, Goimek', location="ES", unit='unit'),
}


In [ ]:
bonsai.search('steel and products thereof')

In [ ]:
bonsai_bio.search('carbon dioxide')

In [ ]:
# export the LCI to Microsoft Excel in an overview format
bi.export.excel.write_lci_excel(bonsai.name, objs=[steel], dirpath=Path.cwd())

#### 4) Defining LCIA functions

In [ ]:
def lca_score(process_dict, method):
    results = []
    for name, process in process_dict.items():
        lca = process.lca(method)
        results.append({
            "Process": name,
            "Location": process.get('location'),
            "Functional Unit": process.get('unit'),
            "Climate Impact (kg CO2eq)": lca.score
        })

    return pd.DataFrame(results)

In [ ]:
def run_contribution_analysis(demand_object,
                              lcia_method,
                              max_level=1,
                              cutoff=0.01):
    return bwa.utils.recursive_calculation_to_object(
        demand_object,
        lcia_method=lcia_method,
        max_level=max_level,
        cutoff=cutoff,
        as_dataframe=True,
    )

In [ ]:
def plot_contribution(df,
                      title='Contribution Analysis',
                      xlabel='CO2 eq. emissions (kg)',
                      drop_parent=True):
    df_plot = df.copy()

    # Drop the parent (the demand itself)
    if drop_parent:
        # Heuristic: parent is usually the one with maximum score
        parent_score = df_plot['score'].max()
        df_plot = df_plot[df_plot['score'] < parent_score]

    # Sort descending by contribution
    df_plot = df_plot.sort_values(by='score', ascending=False)

    f, ax = plt.subplots(figsize=(5, 2))
    sns.barplot(y='name', x='score', data=df_plot, ax=ax)

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=''
    )
    plt.tight_layout()
    plt.show()

#### 5) Calculate and present results

In [ ]:
df_results = lca_score(machining_processes, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_results = lca_score(use_case_machines, config.lcia_gwp100)
display(df_results)

In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

#### 6) Run contribution analysis

In [ ]:
processes_results_ca = {}

for key, demand in machining_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    processes_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
use_case_results_ca = {}

for key, demand in use_case_machines.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    use_case_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )